# AI Report Gen v2 - Component Checks

This notebook allows you to interactively test and verify each component of the research system:
1. **Database Layer**: VectorDBContext (Qdrant + Embeddings)
2. **Agent Layer**: ResearchAgent (LangChain + Gemini)
3. **Pipeline Layer**: ResearchPipeline (Orchestration)

---

## 1. Environment & Setup
Ensure your `.env` file is configured correctly.

In [1]:
import os
import sys
from dotenv import load_dotenv

# Add src to path if needed
sys.path.append(os.path.abspath(".."))

load_dotenv("../.env")

print("Python path updated.")
print(f"GOOGLE_API_KEY set: {bool(os.environ.get('GOOGLE_API_KEY'))}")
print(f"QDRANT_URL set: {os.environ.get('QDRANT_URL', 'Using :memory:')}")

Python path updated.
GOOGLE_API_KEY set: True
QDRANT_URL set: https://3739a4fb-2259-4da5-b652-5997a6b14d62.sa-east-1-0.aws.cloud.qdrant.io:6333


## 2. Database Layer Check
Verify Qdrant connection and embedding generation.

In [3]:
from src.db.database import VectorDBContext

db = VectorDBContext()
db.init_db()

print("Database initialized.")

# # Test saving a dummy report
# test_query = "Testing Qdrant Integration"
# test_report = "# Test Report\nThis is a manual test entry."

# db.save_report(test_query, test_report)
# print("Sample report saved.")

# # Test searching
# cached = db.search_query(test_query)
# print(f"Search result found: {bool(cached)}")
# if cached:
#     print(f"Content preview: {cached[:50]}...")

# Test and see all the reports
print("\n Reports..")
print(db.get_reports())

# # test cleaup
# print("cleanup the collection")
# db.cleanup()


Database initialized.

 Reports..
[]


## 3. Agent Layer Check
Build the ResearchAgent and run a simple query. *Note: This will call the Gemini API.*

In [18]:
from src.agent.core import ResearchAgent

agent = ResearchAgent()
print("Building agent executor...")
agent.build()
print("Agent ready.")

query = "new technology used for bridge construction in india?"
print(f"Invoking agent with query: '{query}'")

# This might take a few seconds
try:
    report = agent.invoke(query)
    print("\n--- GENERATED REPORT ---")
    print(report)
except Exception as e:
    print(f"Error invoking agent: {e}")

Building agent executor...
Agent ready.
Invoking agent with query: 'new technology used for bridge construction in india?'

--- GENERATED REPORT ---
{'messages': [HumanMessage(content='new technology used for bridge construction in india?', additional_kwargs={}, response_metadata={}, id='064b6afd-8015-4e47-8a9c-21cbce862ba4'), AIMessage(content='', additional_kwargs={'function_call': {'name': 'fetch_trends', 'arguments': '{"topic": "new technology bridge construction India", "limit": 5, "source": "google-news"}'}, '__gemini_function_call_thought_signatures__': {'8379f1bf-9f96-4dc5-8cbc-be9472fa75df': 'CtoDAQw51sfRJIFTg4Gil92xru4c9lB+f+kD3wOPDD/meHIpI4ECzYPnaVFzt2B6U6sjUhZ6iRuUYhwYd1OENyhDE1CzFSskfQ9Ozo3FDtFBC5eaWj/ZjuiNwikrOZ93QGIlcRXca8GK/NqkuBx5SCPiRtPb7MszPY9rhJ3dnSqv0k2E8Ur9NsCsDj/+XE/SXghTMajAsHUVNfJiz+qmTkLIqU4J5mFtRbaxFqzsvZ7DVt17unmCD11lm5SkoUQMV0MsQnrSRkswLNXxRxA+2c3RTcJh5QfH9ehvNEuo1bUk7ohdhGSxyLtXLrcFwyXRkBVG3Bg6TDlxvQna2/5Y8uY0cIMP3bSi53abaZeZc7zr0hTnkWM6Gr+eRXXs7wSZTmAVy88

In [19]:
print(report['messages'][-1].content[-1]['text'])

## New Technologies Revolutionizing Bridge Construction in India

### Executive Summary

Bridge construction in India is undergoing a significant transformation driven by the adoption of new technologies, advanced materials, and innovative construction techniques. This shift is leading to faster project completion, enhanced efficiency, and substantial cost reductions. The Indian government's strong focus on infrastructure development is further accelerating this trend, creating a fertile ground for innovation in bridge engineering. Key technological advancements include the widespread implementation of Building Information Modeling (BIM) and the utilization of advanced materials and construction methodologies.

### Detailed Insights

**1. Enhanced Efficiency and Speed through New Technologies:**
New technologies are fundamentally changing the landscape of bridge construction in India. These innovations are streamlining various stages of project execution, from design and planning to ac

## 4. Pipeline Layer Check
Orchestrate a query through the full pipeline (Cache Check -> Agent Generation -> Cache Insertion).

In [15]:
from src.pipeline import ResearchPipeline
import time

pipeline = ResearchPipeline()
pipeline.initialize()

topic = "Data science in 2026"
print(f"Processing query: {topic}")

result = pipeline.process_query(topic)
print(f"Initial status: {result['status']}")

if result['status'] == 'processing':
    task_id = result['task_id']
    print(f"Starting background task: {task_id}")
    
    # Manually run the task (sync for this test)
    pipeline.run_task(task_id, topic)
    
    final_status = pipeline.get_task_status(task_id)
    print(f"Final task status: {final_status['status']}")
    if final_status['report']:
        print("Report successfully generated and cached.")

Processing query: Data science in 2026
Initial status: processing
Starting background task: 1886bfbc-c97d-4022-a34b-db5d409a44e7
Final task status: completed
Report successfully generated and cached.


## 5. Maintenance / Cleanup
View all reports or wipe the database.

In [17]:
reports = pipeline.get_all_reports()
print(f"Total reports in DB: {len(reports)}")
for r in reports:
    print(f"- {r['query']}")
    print(f"- {r['report']}")



Total reports in DB: 2
- Data science in 2026
- {'messages': [{'content': 'Data science in 2026', 'id': '2add90be-e6f4-4d09-b15b-33084cd57589'}, {'content': '', 'additional_kwargs': {'function_call': {'name': 'fetch_trends', 'arguments': '{"source": "podcasts", "topic": "Data science future", "limit": 5}'}, '__gemini_function_call_thought_signatures__': {'402a99fd-df8c-49ea-b25f-93f6f917f824': 'CuwJAQw51sf87AOSMDNfvVykb3N2CtSUgtyDNBBTW/T8p+36zn9iO7SgE6gP42rHtGEjNeg25qPpX71o91hyK+BnyuEawU6tWBwmFWtymA9f3r8GS/2nxnja7TRrUJ+kAFv3Lgmwjm7KKtfGPU75lNsNUYk3CreVM+8RGAjhj2vB360wpV9EwnoQj9Cq2kWOibv2PK60ltlW0ktbecO6jrgpIzqx1gvbYJ/nexYWVmLp21As0ou3caRW8XJSl0Z+hzKtrTRaALyDujIgc4bm6KfuHwV5UnqBn1DcSX2/JWjuEUiziFIvqV0jxex4V7fWIAW7TLyPF5T43Cs2WKRxA0Cfya6YBBtq+oQ4YDVF2rqyI0iDc5tLKjWW5TKotfmHXiibayokkwyO4QcmUqA8JpgiOeORoefAXaW0bN1hNPlGha59Mh1pO0pg1m5DfNlt+fN0Y/FAE+8reQz453zNme4RKxf9Z94m9gTQymJecDE4L6XhuGVpGfrVb7FnJ+WpZRQQDcyXTs9GZyIJXZcq+nDE02pi7bCNsqmxtGezzuvIQjRys5YkEo8P6Fkve/ir1OWe/xyWPeW20Y+CeFrPKCy94B

In [ ]:
# Uncomment to wipe DB
# print("\nCleaning up DB...")
# pipeline.cleanup()
# print("Cleanup complete.")

In [2]:
import asyncio
from src.agent.core import ResearchAgent
from src.db.database import VectorDBContext

async def test_langgraph_agent(query: str = "Compare Python vs JavaScript for AI development"):
    print(f"🚀 Initializing agent for query: '{query}'...\n")
    
    # 1. Initialize Database and Agent
    db = VectorDBContext()
    db.init_db()
    
    agent = ResearchAgent()
    agent.build()
    
    # 2. Run and Stream Output
    print("🔄 Starting research workflow (Streaming steps)...\n")
    
    final_report = ""
    
    # Astream yields steps as the graph progresses
    async for step_data in agent.astream(query):

        step_name = step_data.get('step', 'unknown')
        print(f"  -> {step_name} completed")
        print(step_data)
        
        # Capture the latest content emitted by the nodes
        if step_data.get('content'):
            final_report = step_data['content']
            
    print("\n" + "="*60)
    print("✅ FINAL REPORT GENERATED")
    print("="*60 + "\n")
    print(final_report)
    print("\n" + "="*60)
    
    # 3. Verify Database Cleanup (Ensuring the Cleanup Node worked)
    print("🔍 VERIFYING DATABASE STATE (Cleanup Check)")
    print("="*60)
    
    try:
        todos, _ = db.client.scroll(collection_name="research_todos", limit=10)
        intermediates, _ = db.client.scroll(collection_name="intermediate_reports", limit=10)
        
        print(f"Remaining To-Dos in Qdrant: {len(todos)}")
        print(f"Remaining Intermediate Reports in Qdrant: {len(intermediates)}")
        
        if len(todos) == 0 and len(intermediates) == 0:
            print("✅ Cleanup successful! No intermediate data left behind.")
        else:
            print("⚠️ Warning: Intermediate data was not cleaned up.")
            
    except Exception as e:
        print(f"Could not verify DB state: {e}")

# --- RUN THE TEST ---
# You can execute this directly in a notebook cell:

await test_langgraph_agent("latest data science trends in 2026")

🚀 Initializing agent for query: 'latest data science trends in 2026'...

🔄 Starting research workflow (Streaming steps)...

  -> step: planner completed
{'step': 'step: planner', 'content': '', 'data': {'intent': 'Identify current data science trends', 'goal': 'Provide a concise list of the top data science trends for 2026', 'research_type': 'deep_research', 'todos': [{'id': 't1', 'task': "Search for recent articles, reports, and expert analyses on 'data science trends 2026', 'future of data science', and 'emerging technologies in data science' to gather initial information.", 'status': 'pending'}, {'id': 't2', 'task': 'Analyze the gathered search results to identify recurring themes, key technologies, and significant shifts in the data science landscape projected for 2026.', 'status': 'pending'}, {'id': 't3', 'task': 'Synthesize the identified themes into a concise list of the top 3-5 most prominent data science trends for 2026, including a brief explanation for each.', 'status': 'pen

In [4]:
query = """ 
As AI accelerates demand for high-density compute, data centers are emerging as tightly coupled compute-energy-thermal-water systems, with global electricity demand projected to rise sharply and advanced cooling increasingly required for modern accelerator-heavy workloads. Investigate how next-generation data centers can be designed, built, and operated to optimize performance, availability, energy efficiency, water stewardship, and lifecycle carbon at the same time.
The research should cover site selection, grid interconnection, renewable and dispatchable power sourcing, UPS/battery/generator architecture, rack-density evolution, air vs. liquid cooling, thermal control, workload placement and scheduling, observability, AI-assisted operations, water-use measurement, embodied carbon of buildings and IT hardware, regulatory reporting, supply-chain constraints, capital and operating costs, and long-term scalability. It should also evaluate trade-offs among PUE, WUE, CUE, uptime, latency, resiliency, and total cost of ownership, producing a decision framework for sustainable, AI-ready digital infrastructure.

"""

await test_langgraph_agent(query)

🚀 Initializing agent for query: ' 
As AI accelerates demand for high-density compute, data centers are emerging as tightly coupled compute-energy-thermal-water systems, with global electricity demand projected to rise sharply and advanced cooling increasingly required for modern accelerator-heavy workloads. Investigate how next-generation data centers can be designed, built, and operated to optimize performance, availability, energy efficiency, water stewardship, and lifecycle carbon at the same time.
The research should cover site selection, grid interconnection, renewable and dispatchable power sourcing, UPS/battery/generator architecture, rack-density evolution, air vs. liquid cooling, thermal control, workload placement and scheduling, observability, AI-assisted operations, water-use measurement, embodied carbon of buildings and IT hardware, regulatory reporting, supply-chain constraints, capital and operating costs, and long-term scalability. It should also evaluate trade-offs amo

In [6]:
print('''\n[\n  "Investigate next-generation data center design principles: site selection, grid interconnection, renewable and dispatchable power architecture (UPS, batteries, generators), rack-density evolution, and advanced cooling (air vs. liquid) for high-density AI workloads, including embodied carbon of buildings and IT hardware, capital costs, and supply-chain constraints.",\n  "Explore operational strategies for sustainable AI-ready data centers: workload placement and scheduling, AI-assisted operations, thermal control, observability, and water-use measurement, considering operating costs and regulatory reporting.",\n  "Analyze data center performance trade-offs (PUE, WUE, CUE, uptime, latency, resiliency, TCO) and long-term scalability to develop a decision framework for sustainable, AI-ready digital infrastructure."\n]\n''')


[
  "Investigate next-generation data center design principles: site selection, grid interconnection, renewable and dispatchable power architecture (UPS, batteries, generators), rack-density evolution, and advanced cooling (air vs. liquid) for high-density AI workloads, including embodied carbon of buildings and IT hardware, capital costs, and supply-chain constraints.",
  "Explore operational strategies for sustainable AI-ready data centers: workload placement and scheduling, AI-assisted operations, thermal control, observability, and water-use measurement, considering operating costs and regulatory reporting.",
  "Analyze data center performance trade-offs (PUE, WUE, CUE, uptime, latency, resiliency, TCO) and long-term scalability to develop a decision framework for sustainable, AI-ready digital infrastructure."
]

